# SIGNOVA HISTORICAL PIPELINE NOTICE

## KArSL Label Review and Video QC

Historical notebook containing label investigations, video checks, duplicate detection and QC reporting.

### Execution safety
- Historical research code: **do not run all cells automatically**.
- Review all Google Drive paths before running individual cells.
- Some cells extract archives and create or overwrite CSV reports.
- Some cells delete temporary directories under `/content`.
- Run only in an isolated Colab runtime with the required source files.
- Confirm sufficient disk space and dependencies before processing.

### Reproducibility limitations
- The code is preserved for provenance, not certified as end-to-end reproducible.
- External datasets, original archives and model assets are not included.
- Stored notebook outputs and execution counts have been removed.
- Full MediaPipe extraction across KArSL-502 was not performed in this project.
- The external Pose dataset is a separate third-party source.
- Source licenses, label approvals and video-to-pose identities remain unresolved.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")

text = """# KArSL-502 Label Issues (preliminary)

Status key: CONFIRMED = verified by a reviewer, PROBABLE = strong evidence but not yet verified, OPEN = not investigated.

## 1. S01, sign 5 vs sign 6 (PROBABLE)
- 33 videos stored under sign 5 (recording date 2016-11-10, 16:34:12 to 16:35:04) are probably sign 6.
- Evidence: S01 sign 5 has 83 videos (expected 50) and sign 6 has 17 (expected 50), a difference of +33 / -33.
  The 33 videos form a separate time cluster, 42 s after the other sign-5 recordings, and 17 + 33 = 50.
  Sample frames show four raised fingers in the first cluster and five in the second.
- Handling: original value kept in `sign_id_original` (5). `sign_id_corrected` = 6.
  `qc_status` = REVIEW, `qc_flags` = SUSPECTED_MISFILED_SIGN6.
- Pending: confirmation by a person who knows Arabic Sign Language.

## 2. S01, sign 7 filenames (PROBABLE)
- 33 files in folder 0007 are named with sign ID 0001 (recorded 2016-11-10, 16:35:50 to 16:36:37).
  17 files recorded on 2016-11-17 are named 0007. 33 + 17 = 50, which supports the folder being correct.
- Handling: flagged FILENAME_SIGN_MISMATCH, `qc_status` = REVIEW. Folder ID is used as the sign ID.

## 3. S01, sign 27 has 100 videos (OPEN)
- All 100 videos are from one continuous session on 2016-11-17 (17:15:59 to 17:22:21). No repeated file sizes found.
- Pending: SHA-256 duplicate check from the full video QC.

## 4. Signs with 25 videos instead of 50 (OPEN)
- S01: signs 348, 363, 364. S03: signs 163, 175, 184, 387, 388.
- Treated as missing data, not removed. To be noted in the dataset card.

## 5. Other count differences (OPEN)
- S02 sign 499 (+18), S03 sign 59 (+10), plus smaller differences listed in count_anomalies.csv.

## 6. Label text issues (OPEN)
- Duplicate English labels: signs 213 and 214 (nephew), signs 104 and 355 (thermometer). Use SignID, not English text, as the class key.
"""
(R / "label_issues.md").write_text(text, encoding="utf-8")
print("saved")

In [ ]:
import pandas as pd, subprocess
from pathlib import Path
from IPython.display import Video, display

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
df = pd.read_csv(R / "metadata_karsl_canonical.csv",
                 dtype={"signer": str}, low_memory=False)

g = df[df.qc_flags == "SUSPECTED_MISFILED_SIGN6"].sort_values("sample_id")
print("suspected videos:", len(g))          # لازم يطلع 33
picks = g.iloc[[0, len(g)//2, -1]]

tmp = Path("/content/view")
tmp.mkdir(exist_ok=True)

for _, r in picks.iterrows():
    subprocess.run(
        ["7z", "e", r.archive_path, f"-o{tmp}", r.video_path_in_archive,
         "-y", "-bso0", "-bsp0"],
        check=True
    )
    p = tmp / Path(r.video_path_in_archive).name
    print(p.name)
    display(Video(str(p), embed=True, width=480))
    ref = df[(df.signer_id == "S01") & (df.sign_id_original == 5) &
         (df.recording_date == "16-11-10") &
         (df.qc_flags != "SUSPECTED_MISFILED_SIGN6")].sort_values("sample_id")
print("reference videos:", len(ref))
for _, r in ref.iloc[[0, len(ref)//2, -1]].iterrows():
    subprocess.run(
        ["7z", "e", r.archive_path, f"-o{tmp}", r.video_path_in_archive,
         "-y", "-bso0", "-bsp0"],
        check=True
    )
    p = tmp / Path(r.video_path_in_archive).name
    print(p.name)
    display(Video(str(p), embed=True, width=480))

In [ ]:
from pathlib import Path
R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
p = R / "label_issues.md"
t = p.read_text(encoding="utf-8")

t = t.replace(
"## 1. S01, sign 5 vs sign 6 (PROBABLE)",
"## 1. S01, sign 5 vs sign 6 (VISUALLY CONFIRMED, expert review pending)")

t = t.replace(
"- Pending: confirmation by a person who knows Arabic Sign Language.",
"- Visual check (team member, 2026-10-15): 3 full videos from the 41-video cluster show four raised fingers "
"(number 4, sign 5); 3 full videos from the 33-video cluster show five spread fingers (number 5, sign 6).\n"
"- Pending: confirmation by a person who knows Arabic Sign Language. The correction stays in `sign_id_corrected`; "
"the original value is never overwritten.")

p.write_text(t, encoding="utf-8")
print(t[:1500])

In [ ]:
from pathlib import Path

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
parts = sorted((R / "video_qc_parts").glob("*.csv"))
print(len(parts), "/ 30")
for p in parts:
    print(" ", p.name)

In [ ]:
from pathlib import Path
R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
p = R / "label_issues.md"
t = p.read_text(encoding="utf-8")

# العنوان
t = t.replace(
"## 1. S01, sign 5 vs sign 6 (VISUALLY CONFIRMED, expert review pending)",
"## 1. S01, sign 5 vs sign 6 (VISUALLY CHECKED BY TEAM MEMBER, no sign-language expert available)")
t = t.replace(
"## 1. S01, sign 5 vs sign 6 (PROBABLE)",
"## 1. S01, sign 5 vs sign 6 (VISUALLY CHECKED BY TEAM MEMBER, no sign-language expert available)")

# سطر الانتظار
t = t.replace(
"- Pending: confirmation by a person who knows Arabic Sign Language. The correction stays in `sign_id_corrected`; the original value is never overwritten.",
"- Limitation: no sign-language expert reviewed this. The decision rests on (a) the recording-time gap, "
"(b) the count arithmetic 17 + 33 = 50, and (c) a visual check of the number of raised fingers.\n"
"- Handling: `sign_id_corrected` = 6 for these 33 videos. `sign_id_original` (5) is never overwritten. "
"`qc_status` = REVIEW. ML users may include or exclude REVIEW samples with one filter.")
t = t.replace(
"- Pending: confirmation by a person who knows Arabic Sign Language.",
"- Limitation: no sign-language expert reviewed this. The decision rests on (a) the recording-time gap, "
"(b) the count arithmetic 17 + 33 = 50, and (c) a visual check of the number of raised fingers.\n"
"- Handling: `sign_id_corrected` = 6 for these 33 videos. `sign_id_original` (5) is never overwritten. "
"`qc_status` = REVIEW. ML users may include or exclude REVIEW samples with one filter.")

p.write_text(t, encoding="utf-8")
print(t[:1800])

In [ ]:
from pathlib import Path
R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
p = R / "label_issues.md"
t = p.read_text(encoding="utf-8")

old = ("- Handling: original value kept in `sign_id_original` (5). `sign_id_corrected` = 6.\n"
       "  `qc_status` = REVIEW, `qc_flags` = SUSPECTED_MISFILED_SIGN6.\n")
t = t.replace(old, "")

t = t.replace(
"Status key: CONFIRMED = verified by a reviewer, PROBABLE = strong evidence but not yet verified, OPEN = not investigated.",
"Status key: CONFIRMED = verified by an expert reviewer, VISUALLY CHECKED = checked by a team member without expert review, "
"PROBABLE = strong evidence but not yet verified, OPEN = not investigated.")

t = t.replace("`sign_id_original` (5) is never overwritten. `qc_status` = REVIEW.",
              "`sign_id_original` (5) is never overwritten. `qc_status` = REVIEW, `qc_flags` = SUSPECTED_MISFILED_SIGN6.")

p.write_text(t, encoding="utf-8")
print(t[:1700])

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")

# 1) تجميع الأجزاء
parts = sorted((R / "video_qc_parts").glob("*.csv"))
print("parts:", len(parts))
qc = pd.concat([pd.read_csv(p) for p in parts], ignore_index=True)
print("qc rows:", len(qc), "| unique sample_id:", qc.sample_id.nunique())
qc.to_csv(R / "video_qc_full.csv", index=False)

# 2) الدمج مع الـ metadata
meta = pd.read_csv(R / "metadata_karsl_canonical.csv", dtype={"signer": str}, low_memory=False)
df = meta.merge(qc, on="sample_id", how="left", validate="one_to_one")
df["qc_flags"] = df["qc_flags"].fillna("")
print("videos without QC row:", df.decode_ok.isna().sum() if "decode_ok" in df else "no decode_ok column")

# 3) العلامات
def add(mask, flag):
    df.loc[mask, "qc_flags"] = df.loc[mask, "qc_flags"].apply(lambda s: flag if s == "" else s + ";" + flag)

missing = df.decode_ok.isna() | (df.get("missing", False) == True)
add(missing, "MISSING_FILE")
add(~missing & (df.decode_ok == False), "DECODE_FAIL")
add(~missing & (df.decode_ok == True) & (df.last_frame_ok == False), "LAST_FRAME_FAIL")

SHORT_FRAMES, LONG_SEC, FPS_TOL = 15, 10, 1.0
ok = ~missing & (df.decode_ok == True)
add(ok & (df.frames < SHORT_FRAMES), "TOO_SHORT")
add(ok & (df.duration_sec > LONG_SEC), "TOO_LONG")
add(ok & ((df.fps - 30).abs() > FPS_TOL), "FPS_ODD")
mode_res = df.loc[ok].groupby(["width", "height"]).size().idxmax()
print("most common resolution:", mode_res)
add(ok & ((df.width != mode_res[0]) | (df.height != mode_res[1])), "RESOLUTION_ODD")

dup = df.sha256.notna() & df.duplicated("sha256", keep=False)
add(dup, "DUPLICATE_HASH")

# 4) الحالة النهائية
EXCLUDE_FLAGS = ("MISSING_FILE", "DECODE_FAIL")
def status(f):
    if f == "": return "PASS"
    if any(x in f for x in EXCLUDE_FLAGS): return "EXCLUDE"
    return "REVIEW"
df["qc_status"] = df.qc_flags.apply(status)
df["qc_reviewer"] = np.where(df.qc_status == "PASS", "", "auto")

# 5) ملخص
print("\n=== qc_status ===");  print(df.qc_status.value_counts())
print("\n=== flags ===");      print(df.qc_flags[df.qc_flags != ""].str.split(";").explode().value_counts())
print("\n=== duration_sec ==="); print(df.duration_sec.describe())
print("\n=== frames ===");      print(df.frames.describe())
print("\n=== fps ===");         print(df.fps.round(1).value_counts().head())
print("\n=== resolution ===");  print(df.groupby(["width","height"]).size())

# 6) التكرارات
d = df[dup].sort_values(["sha256", "sample_id"])
print("\nvideos in duplicate groups:", len(d), "| groups:", d.sha256.nunique())
if len(d):
    g = d.groupby("sha256").agg(n=("sample_id","size"), splits=("split","nunique"),
                                signers=("signer_id","nunique"), signs=("sign_id","nunique"))
    print("groups crossing train/test:", (g.splits > 1).sum())
    print("groups crossing signers:", (g.signers > 1).sum())
    print("groups with different signs:", (g.signs > 1).sum())
    d[["sample_id","sha256","signer_id","split","sign_id","video_path_in_archive"]] \
        .to_csv(R / "duplicate_report.csv", index=False, encoding="utf-8-sig")

# 7) sign 27 عند S01
s27 = df[(df.signer_id == "S01") & (df.sign_id == 27)]
print("\nS01 sign 27: videos =", len(s27), "| unique hashes =", s27.sha256.nunique())

# 8) الحفظ (ملفات جديدة، القديمة مش بتتلمس)
df.to_csv(R / "metadata_karsl_qc.csv", index=False, encoding="utf-8-sig")
cols = ["sample_id","qc_status","qc_flags","qc_reviewer","signer_id","sign_id","split","video_path_in_archive"]
df[df.qc_status != "PASS"][cols].to_csv(R / "qc_flags.csv", index=False, encoding="utf-8-sig")
print("\nsaved: metadata_karsl_qc.csv, qc_flags.csv, video_qc_full.csv, duplicate_report.csv")

In [ ]:
import pandas as pd
from pathlib import Path

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
df = pd.read_csv(R / "metadata_karsl_qc.csv", dtype={"signer": str}, low_memory=False)

d = df[df.qc_flags.str.contains("DUPLICATE_HASH", na=False)].sort_values(["sha256", "sample_id"]).copy()
d["k"] = d.groupby("sha256").cumcount()
a = d[d.k == 0].set_index("sha256")
b = d[d.k == 1].set_index("sha256")
pairs = a[["signer_id", "sign_id", "split", "name_sign", "recording_date", "video_path_in_archive"]].join(
    b[["sign_id", "split", "name_sign", "video_path_in_archive"]], rsuffix="_2")

print("=== pairs by (signer, sign_a, sign_b) ===")
print(pairs.groupby(["signer_id", "sign_id", "sign_id_2"]).size())
print("\n=== pairs by (split_a, split_b) ===")
print(pairs.groupby(["split", "split_2"]).size())
print("\n=== sample pairs ===")
display(pairs.head(8))

print("\n=== frames quantiles ===")
print(df.frames.quantile([.001, .01, .05, .10]))
print("frames < 10:", (df.frames < 10).sum(), "| frames < 15:", (df.frames < 15).sum())

df["short"] = df.qc_flags.str.contains("TOO_SHORT", na=False)
print("\n=== short ratio per signer ===")
print(df.groupby("signer_id").short.mean().round(4))

s = df.groupby(["sign_id", "Sign-English"]).short.agg(["sum", "mean"])
print("\n=== signs with most short videos ===")
print(s[s["sum"] > 0].sort_values("sum", ascending=False).head(15))

In [ ]:
import pandas as pd
from pathlib import Path

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
df = pd.read_csv(R / "metadata_karsl_qc.csv", dtype={"signer": str}, low_memory=False)

print("=== videos per signer and split ===")
print(df.groupby(["signer_id", "split"]).size().unstack(fill_value=0))

print("\n=== videos per class: signer breakdown for odd classes ===")
counts = df.groupby("sign_id").size()
odd = counts[(counts != 150)].index
t = df[df.sign_id.isin(odd)].groupby(["sign_id", "signer_id"]).size().unstack(fill_value=0)
t["total"] = t.sum(axis=1)
display(t.sort_values("total").head(15))
display(t.sort_values("total").tail(8))

print("\n=== signs 5, 6, 27 by signer and split ===")
display(df[df.sign_id.isin([5, 6, 27])].groupby(["sign_id", "signer_id", "split"]).size().unstack(fill_value=0))

print("\n=== qc flags summary ===")
print(df.qc_flags.fillna("NONE").str.split(",").explode().value_counts())

In [ ]:
import pandas as pd
from pathlib import Path

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
df = pd.read_csv(R / "metadata_karsl_qc.csv", dtype={"signer": str}, low_memory=False)

ts = df.video_path_in_archive.str.extract(r"\((\d{2}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2})\)")[0]
df["ts"] = pd.to_datetime(ts, format="%y_%m_%d_%H_%M_%S", errors="coerce")

s1 = df[(df.signer_id == "S01") & df.sign_id.between(25, 30)].copy()
s1["dup"] = s1.qc_flags.str.contains("DUPLICATE_HASH", na=False)
print("=== S01 signs 25-30 timestamps ===")
print(s1.groupby(["sign_id", "dup"]).ts.agg(["count", "min", "max"]))

m6 = df[df.qc_flags.str.contains("SUSPECTED_MISFILED_SIGN6", na=False)]
print("\n=== suspected misfiled sign 6 ===")
print(m6.groupby(["signer_id", "sign_id", "name_sign", "split"]).size())

mm = df[df.qc_flags.str.contains("FILENAME_SIGN_MISMATCH", na=False)]
print("\n=== filename sign mismatch ===")
print(mm.groupby(["signer_id", "sign_id", "name_sign", "split"]).size())

print("\n=== videos under 10 frames ===")
print(df[df.frames < 10][["sample_id", "signer_id", "sign_id", "Sign-English", "frames", "video_path_in_archive"]])

s2 = df[(df.signer_id == "S02") & df.qc_flags.str.contains("TOO_SHORT", na=False)]
print("\n=== S02 short videos: share in sign ids 32-59 ===")
print(len(s2), round(s2.sign_id.between(32, 59).mean(), 3))

In [ ]:
import pandas as pd
from pathlib import Path

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
df = pd.read_csv(R / "metadata_karsl_qc.csv", dtype={"signer": str}, low_memory=False)
ts = df.video_path_in_archive.str.extract(r"\((\d{2}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2})\)")[0]
df["ts"] = pd.to_datetime(ts, format="%y_%m_%d_%H_%M_%S", errors="coerce")
df["mm"] = df.qc_flags.str.contains("FILENAME_SIGN_MISMATCH", na=False)

print(df[df.mm][["sample_id", "sign_id", "name_sign", "split", "ts", "video_path_in_archive"]].head(8).to_string())

print("\n=== signs 1 and 7 count per signer ===")
print(df[df.sign_id.isin([1, 7])].groupby(["sign_id", "signer_id"]).size().unstack(fill_value=0))

print("\n=== S01 timestamps ===")
s = df[(df.signer_id == "S01") & df.sign_id.isin([1, 7])]
print(s.groupby(["sign_id", "mm"]).ts.agg(["count", "min", "max"]))

print("\n=== S02 short videos outside 32-59 ===")
s2 = df[(df.signer_id == "S02") & df.qc_flags.str.contains("TOO_SHORT", na=False) & ~df.sign_id.between(32, 59)]
print(s2.groupby(["sign_id", "Sign-English"]).size().sort_values(ascending=False).head(15))

In [ ]:
import subprocess, shutil, cv2, numpy as np
import pandas as pd
from pathlib import Path

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
df = pd.read_csv(R / "metadata_karsl_qc.csv", dtype={"signer": str}, low_memory=False)
df["dup"] = df.qc_flags.str.contains("DUPLICATE_HASH", na=False)
TMP = Path("/content/view_tmp")

picks = [
    ("S01 sign27 dup", df[(df.signer_id == "S01") & (df.sign_id == 27) & df.dup].iloc[0]),
    ("S01 sign27 unique", df[(df.signer_id == "S01") & (df.sign_id == 27) & ~df.dup].iloc[0]),
    ("S01 sign28 dup", df[(df.signer_id == "S01") & (df.sign_id == 28)].iloc[0]),
    ("S02 sign27", df[(df.signer_id == "S02") & (df.sign_id == 27)].iloc[0]),
    ("S02 sign28", df[(df.signer_id == "S02") & (df.sign_id == 28)].iloc[0]),
    ("S03 sign27", df[(df.signer_id == "S03") & (df.sign_id == 27)].iloc[0]),
    ("S03 sign28", df[(df.signer_id == "S03") & (df.sign_id == 28)].iloc[0]),
]

def strip(path, n=6, w=200):
    cap = cv2.VideoCapture(str(path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    out = []
    for i in np.linspace(0, total - 1, n).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, f = cap.read()
        if not ok:
            f = np.zeros((1080, 1920, 3), np.uint8)
        h = int(f.shape[0] * w / f.shape[1])
        out.append(cv2.resize(f, (w, h)))
    cap.release()
    return np.hstack(out)


rows = []
for label, r in picks:
    shutil.rmtree(TMP, ignore_errors=True); TMP.mkdir(parents=True)
    subprocess.run(["7z", "x", r.archive_path, f"-o{TMP}", "-y", "-bso0", "-bsp0", r.video_path_in_archive], check=True)
    img = strip(TMP / r.video_path_in_archive)
    cv2.putText(img, label, (5, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    rows.append(img)

montage = np.vstack(rows)
cv2.imwrite(str(R / "sign27_28_check.png"), montage)
from google.colab.patches import cv2_imshow
cv2_imshow(montage)

In [ ]:
import subprocess, shutil, cv2, numpy as np
import pandas as pd
from pathlib import Path

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
df = pd.read_csv(R / "metadata_karsl_qc.csv", dtype={"signer": str}, low_memory=False)
df["dup"] = df.qc_flags.str.contains("DUPLICATE_HASH", na=False)
df["mm"] = df.qc_flags.str.contains("FILENAME_SIGN_MISMATCH", na=False)
TMP = Path("/content/view_tmp2")

def pick(signer, sign, dup=None, mm=None, k=0):
    d = df[(df.signer_id == signer) & (df.sign_id == sign)]
    if dup is not None:
        d = d[d.dup == dup]
    if mm is not None:
        d = d[d.mm == mm]
    return d.iloc[k]

def strip(path, label):
    cap = cv2.VideoCapture(str(path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    out = []
    for q in (0.25, 0.4, 0.55, 0.7):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(total * q))
        ok, f = cap.read()
        if not ok:
            f = np.zeros((1080, 1920, 3), np.uint8)
        out.append(cv2.resize(f[100:800, 700:1400], (250, 250)))
    cap.release()
    img = np.hstack(out)
    cv2.putText(img, label, (5, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    return img

def build(picks, name):
    rows = []
    for label, r in picks:
        shutil.rmtree(TMP, ignore_errors=True); TMP.mkdir(parents=True)
        subprocess.run(["7z", "x", r.archive_path, f"-o{TMP}", "-y", "-bso0", "-bsp0", r.video_path_in_archive], check=True)
        rows.append(strip(TMP / r.video_path_in_archive, label))
    m = np.vstack(rows)
    cv2.imwrite(str(R / name), m)
    from google.colab.patches import cv2_imshow
    cv2_imshow(m)

build([
    ("S01 sign27 dup", pick("S01", 27, dup=True)),
    ("S01 sign27 unique", pick("S01", 27, dup=False)),
    ("S01 sign28 copy", pick("S01", 28)),
    ("S02 sign27", pick("S02", 27)),
    ("S02 sign28", pick("S02", 28)),
    ("S03 sign27", pick("S03", 27)),
    ("S03 sign28", pick("S03", 28)),
], "sign27_28_big.png")

build([
    ("S01 sign7 mismatch", pick("S01", 7, mm=True)),
    ("S01 sign7 normal", pick("S01", 7, mm=False)),
    ("S01 sign1", pick("S01", 1)),
    ("S02 sign7", pick("S02", 7)),
    ("S02 sign1", pick("S02", 1)),
], "sign7_check.png")

In [ ]:
import pandas as pd
from pathlib import Path

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
df = pd.read_csv(R / "metadata_karsl_qc.csv", dtype={"signer": str}, low_memory=False)
flags = df.qc_flags.fillna("")

df["sign_id_corrected"] = df.sign_id
df["qc_status"] = "PASS"
df["qc_reason"] = ""
df["usable"] = True

def mark(mask, status, reason, usable):
    old = df.loc[mask, "qc_reason"]
    df.loc[mask, "qc_reason"] = old.where(old == "", old + ";") + reason
    df.loc[mask, "qc_status"] = status
    df.loc[mask, "usable"] = usable

short = flags.str.contains("TOO_SHORT")
mm = flags.str.contains("FILENAME_SIGN_MISMATCH")
m6 = flags.str.contains("SUSPECTED_MISFILED_SIGN6")
tiny = df.frames < 10
quar = (df.signer_id == "S01") & df.sign_id.isin([27, 28])

mark(short, "PASS", "SHORT_SIGNER_STYLE", True)
mark(mm, "REVIEW", "FILENAME_WRONG_LABEL_7_KEPT_BY_GESTURE", True)
df.loc[m6, "sign_id_corrected"] = 6
mark(m6, "REVIEW", "LABEL_CORRECTED_5_TO_6", True)
mark(tiny, "REVIEW", "UNDER_10_FRAMES_MANUAL_CHECK", False)
mark(quar, "EXCLUDE", "QUARANTINE_S01_SIGN27_28_UNRESOLVED", False)

print(df.qc_status.value_counts())

print("\nusable per signer")
print(df[df.usable].groupby("signer_id").size())
print("\nusable per corrected sign, odd classes")
t = df[df.usable & df.sign_id_corrected.isin([5, 6, 7, 27, 28])]
print(t.groupby(["sign_id_corrected", "signer_id"]).size().unstack(fill_value=0))
print("\nclasses with usable count != 150")
c = df[df.usable].groupby("sign_id_corrected").size()
print(c[c != 150].describe())
print((c != 150).sum())

df.to_csv(R / "metadata_karsl_corrected.csv", index=False)
print("saved", len(df))

In [ ]:

import pandas as pd
from pathlib import Path

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")

df = pd.read_csv(
    R / "metadata_karsl_corrected.csv",
    dtype={"signer_id": str},
    low_memory=False
)

print("=== 1. QC Status ===")
print(df["qc_status"].value_counts(dropna=False))

print("\n=== 2. Usable Samples ===")
print(df["usable"].value_counts(dropna=False))

print("\n=== 3. QC Reason Summary ===")
print(
    df["qc_reason"]
    .fillna("")
    .str.split(";")
    .explode()
    .loc[lambda s: s.ne("")]
    .value_counts()
)

print("\n=== 4. Status vs Usability ===")
print(pd.crosstab(df["qc_status"], df["usable"]))

print("\n=== 5. Corrected Sign Counts ===")
counts = df[df["usable"]].groupby(
    "sign_id_corrected"
).size()

print("Usable classes:", counts.index.nunique())
print("Minimum samples per class:", counts.min())
print("Maximum samples per class:", counts.max())
print("Classes with fewer than 150 samples:", (counts < 150).sum())
print("Classes with more than 150 samples:", (counts > 150).sum())

print("\n=== 6. Review Cases: Signs 5, 6 and 7 ===")
review_ids = df["sign_id"].isin([5, 6, 7])

display(
    df.loc[
        review_ids,
        [
            "sample_id",
            "signer_id",
            "split",
            "sign_id",
            "sign_id_corrected",
            "qc_status",
            "qc_reason",
            "usable"
        ]
    ].groupby(
        [
            "signer_id",
            "split",
            "sign_id",
            "sign_id_corrected",
            "qc_status",
            "qc_reason",
            "usable"
        ],
        dropna=False
    ).size().reset_index(name="video_count")
)

print("\n=== Audit complete: no files modified ===")


In [ ]:

# Audit excluded samples without modifying any file

excluded = df[df["qc_status"] == "EXCLUDE"].copy()

print("Excluded samples:", len(excluded))

print("\n=== Exclusion reasons ===")
print(excluded["qc_reason"].value_counts(dropna=False))

print("\n=== Excluded samples by split ===")
print(excluded["split"].value_counts(dropna=False))

print("\n=== Excluded samples by sign ===")
display(
    excluded.groupby(["sign_id", "sign_id_corrected"])
    .size()
    .reset_index(name="excluded_count")
    .sort_values("excluded_count", ascending=False)
)

print("\n=== Usability and status mismatches ===")
display(
    df[
        ((df["qc_status"] == "EXCLUDE") & (df["usable"] == True))
        | ((df["qc_status"] == "PASS") & (df["usable"] == False))
    ][
        ["sample_id", "signer_id", "split", "sign_id",
         "sign_id_corrected", "qc_status", "qc_reason", "usable"]
    ].head(30)
)

print("\nAudit only — no files modified.")


In [ ]:

import os

reports_path = "/content/drive/MyDrive/SIGNOVA/KArSL/reports"

for file in sorted(os.listdir(reports_path)):
    print(file)


In [ ]:

import os

reports_path = "/content/drive/MyDrive/SIGNOVA/KArSL/reports"

keywords = ["duplicate", "hash", "quarantine", "qc", "corrected"]

for file in sorted(os.listdir(reports_path)):
    if any(word in file.lower() for word in keywords):
        print(file)


In [ ]:

import os
import pandas as pd

reports_path = "/content/drive/MyDrive/SIGNOVA/KArSL/reports"
file_path = os.path.join(reports_path, "duplicate_report.csv")

duplicates = pd.read_csv(file_path)

print("Shape:", duplicates.shape)

print("\n=== Columns ===")
print(duplicates.columns.tolist())

print("\n=== First 10 rows ===")
display(duplicates.head(10))

print("\n=== Missing values ===")
display(duplicates.isna().sum())


In [ ]:
print("Total rows:", len(duplicates))
print("Unique SHA256 hashes:", duplicates["sha256"].nunique())

hash_counts = (
    duplicates.groupby("sha256")
    .agg(
        copies=("sample_id", "count"),
        signs=("sign_id", "nunique"),
        splits=("split", "nunique"),
        sample_ids=("sample_id", lambda x: list(x)),
        sign_ids=("sign_id", lambda x: sorted(x.unique())),
        split_names=("split", lambda x: sorted(x.unique()))
    )
    .reset_index()
)

print("\n=== Hash groups with multiple copies ===")
display(hash_counts[hash_counts["copies"] > 1])

cross_sign = hash_counts[hash_counts["signs"] > 1].copy()

print("\nGroups duplicated across different signs:", len(cross_sign))
display(cross_sign)

cross_split = hash_counts[hash_counts["splits"] > 1].copy()

print("\nGroups appearing in both train and test:", len(cross_split))
display(cross_split)

print("\nAudit only — no files modified.")

In [ ]:
import os
import pandas as pd

reports_path = "/content/drive/MyDrive/SIGNOVA/KArSL/reports"

duplicates = pd.read_csv(
    os.path.join(reports_path, "duplicate_report.csv")
)

qc = pd.read_csv(
    os.path.join(reports_path, "metadata_karsl_corrected.csv")
)

excluded = qc[qc["qc_status"] == "EXCLUDE"].copy()

duplicate_ids = set(duplicates["sample_id"].astype(str))
excluded_ids = set(excluded["sample_id"].astype(str))

print("Duplicate report rows:", len(duplicates))
print("Unique samples in duplicate report:", len(duplicate_ids))
print("Excluded samples:", len(excluded_ids))
print("Excluded samples listed in duplicate report:",
      len(excluded_ids & duplicate_ids))
print("Excluded samples missing from duplicate report:",
      len(excluded_ids - duplicate_ids))

print("\n=== Excluded samples missing from duplicate report ===")
display(
    excluded[excluded["sample_id"].astype(str).isin(
        excluded_ids - duplicate_ids
    )][
        ["sample_id", "signer_id", "split", "sign_id",
         "sign_id_corrected", "qc_status", "qc_reason", "usable"]
    ].head(30)
)

print("\nAudit only — no files modified.")

In [ ]:
import os
import pandas as pd

reports_path = "/content/drive/MyDrive/SIGNOVA/KArSL/reports"

qc = pd.read_csv(
    os.path.join(reports_path, "metadata_karsl_corrected.csv"),
    low_memory=False
)

duplicates = pd.read_csv(
    os.path.join(reports_path, "duplicate_report.csv")
)

video_qc = pd.read_csv(
    os.path.join(reports_path, "video_qc_full.csv"),
    low_memory=False
)

excluded = qc[qc["qc_status"] == "EXCLUDE"].copy()

duplicate_ids = set(duplicates["sample_id"].astype(str))
missing = excluded[
    ~excluded["sample_id"].astype(str).isin(duplicate_ids)
].copy()

print("=== Missing excluded samples ===")
print("Total:", len(missing))

print("\n=== Counts by sign and split ===")
display(
    missing.groupby(
        ["sign_id", "sign_id_corrected", "split"]
    ).size().reset_index(name="count")
)

print("\n=== Video QC columns ===")
print(video_qc.columns.tolist())

print("\n=== First 20 missing samples ===")
display(
    missing[
        ["sample_id", "signer_id", "split", "sign_id",
         "sign_id_corrected", "qc_reason", "usable"]
    ].head(20)
)

print("\nAudit only — no files modified.")

In [ ]:
import os
import pandas as pd

reports_path = "/content/drive/MyDrive/SIGNOVA/KArSL/reports"

qc = pd.read_csv(
    os.path.join(reports_path, "metadata_karsl_corrected.csv"),
    low_memory=False
)

video_qc = pd.read_csv(
    os.path.join(reports_path, "video_qc_full.csv"),
    low_memory=False
)

print("=== Corrected metadata columns ===")
print(qc.columns.tolist())

print("\n=== Video QC columns ===")
print(video_qc.columns.tolist())

print("\n=== Hash-related columns after merge ===")
excluded = qc[qc["qc_status"] == "EXCLUDE"].copy()

excluded["sample_id"] = excluded["sample_id"].astype(str)
video_qc["sample_id"] = video_qc["sample_id"].astype(str)

excluded_hashes = excluded.merge(
    video_qc[["sample_id", "sha256", "decode_ok", "frames"]],
    on="sample_id",
    how="left",
    suffixes=("_metadata", "_video_qc"),
    validate="one_to_one"
)

print([
    col for col in excluded_hashes.columns
    if "sha256" in col.lower()
])

print("\nAudit only — no files modified.")

In [ ]:
import os
import pandas as pd

reports_path = "/content/drive/MyDrive/SIGNOVA/KArSL/reports"

qc = pd.read_csv(
    os.path.join(reports_path, "metadata_karsl_corrected.csv"),
    low_memory=False
)

video_qc = pd.read_csv(
    os.path.join(reports_path, "video_qc_full.csv"),
    low_memory=False
)

excluded = qc[qc["qc_status"] == "EXCLUDE"].copy()
excluded["sample_id"] = excluded["sample_id"].astype(str)
video_qc["sample_id"] = video_qc["sample_id"].astype(str)

# Keep only the video-QC columns needed for this audit.
excluded_hashes = excluded.drop(
    columns=["sha256", "decode_ok", "frames"],
    errors="ignore"
).merge(
    video_qc[["sample_id", "sha256", "decode_ok", "frames"]],
    on="sample_id",
    how="left",
    validate="one_to_one"
)

hash_groups = (
    excluded_hashes.groupby("sha256")
    .agg(
        copies=("sample_id", "count"),
        sample_ids=("sample_id", list),
        signs=("sign_id", lambda x: sorted(x.unique())),
        splits=("split", lambda x: sorted(x.unique()))
    )
    .reset_index()
)

print("Excluded samples:", len(excluded_hashes))
print("Missing hashes:", excluded_hashes["sha256"].isna().sum())
print("Unique hashes:", excluded_hashes["sha256"].nunique())
print("Hashes occurring once:", (hash_groups["copies"] == 1).sum())
print("Hashes occurring twice:", (hash_groups["copies"] == 2).sum())
print("Hashes occurring more than twice:", (hash_groups["copies"] > 2).sum())

print("\n=== Samples with hashes occurring only once ===")
display(
    excluded_hashes[
        excluded_hashes["sha256"].isin(
            hash_groups.loc[hash_groups["copies"] == 1, "sha256"]
        )
    ][
        ["sample_id", "sign_id", "split", "decode_ok", "frames", "sha256"]
    ]
)

print("\nAudit only — no files modified.")

In [ ]:
import os
import pandas as pd

reports_path = "/content/drive/MyDrive/SIGNOVA/KArSL/reports"

qc = pd.read_csv(
    os.path.join(reports_path, "metadata_karsl_corrected.csv"),
    low_memory=False
)

video_qc = pd.read_csv(
    os.path.join(reports_path, "video_qc_full.csv"),
    low_memory=False
)

excluded = qc[qc["qc_status"] == "EXCLUDE"].copy()

excluded["sample_id"] = excluded["sample_id"].astype(str)
video_qc["sample_id"] = video_qc["sample_id"].astype(str)

excluded = excluded.drop(
    columns=["sha256", "decode_ok", "frames"],
    errors="ignore"
).merge(
    video_qc[["sample_id", "sha256", "decode_ok", "frames"]],
    on="sample_id",
    how="left",
    validate="one_to_one"
)

hash_counts = excluded["sha256"].value_counts()
unique_hashes = hash_counts[hash_counts == 1].index

review_manifest = excluded[
    excluded["sha256"].isin(unique_hashes)
].copy()

review_manifest = review_manifest[
    [
        "sample_id", "signer_id", "split",
        "sign_id", "sign_id_corrected",
        "frames", "decode_ok", "sha256",
        "video_path_in_archive", "archive_path",
        "qc_reason"
    ]
].sort_values(["split", "sample_id"])

output_path = os.path.join(
    reports_path, "sign27_unique_quarantine_review.csv"
)

review_manifest.to_csv(output_path, index=False)

print("Unique-hash samples for review:", len(review_manifest))
print("Manifest saved to:", output_path)
display(review_manifest.head(10))

print("\nAudit only — original QC files unchanged.")

In [ ]:
import pandas as pd

review = pd.read_csv(
    "/content/drive/MyDrive/SIGNOVA/KArSL/reports/sign27_unique_quarantine_review.csv"
)

print("Total samples:", len(review))
print("\nSamples by split:")
print(review["split"].value_counts())

print("\nFrame-count summary:")
print(review["frames"].describe())

print("\nDecode status:")
print(review["decode_ok"].value_counts(dropna=False))

print("\nSign IDs:")
print(review["sign_id"].value_counts(dropna=False))

In [ ]:
import pandas as pd
from pathlib import Path

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
df = pd.read_csv(R / "metadata_karsl_corrected.csv", low_memory=False)
u = df[df.usable].copy()

print("usable:", len(u), "| duplicate hashes among usable:", int(u.sha256.duplicated().sum()))

sm = u[["sample_id", "signer_id", "session_id", "sign_id_corrected", "split", "qc_status"]].rename(
    columns={"split": "author_split", "sign_id_corrected": "sign_id"})
for s in ["S01", "S02", "S03"]:
    sm["loso_test_" + s] = (sm.signer_id == s).map({True: "test", False: "train"})
sm["split_version"] = "v2"
sm.to_csv(R / "split_manifest_v2.csv", index=False)

print(sm.author_split.value_counts().to_dict())
for s in ["S01", "S02", "S03"]:
    t = sm[sm.signer_id == s]
    print(s, "test samples:", len(t), "| classes:", t.sign_id.nunique(), "| missing classes:", sorted(set(range(1, 503)) - set(t.sign_id)))

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import date

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
df = pd.read_csv(R / "metadata_karsl_corrected.csv", low_memory=False)
df["qc_flags"] = df.qc_flags.fillna("")
df["qc_reason"] = df.qc_reason.fillna("")
u = df[df.usable]
today = date.today().isoformat()

for n in ["qc_report.md", "dataset_card.md", "handoff.md"]:
    p = R / n
    if p.exists():
        p.rename(R / n.replace(".md", "_prelim.md"))

orig = pd.read_excel("/content/drive/MyDrive/KArSL-502/KARSL-502_Labels.xlsx")
rev = df.groupby("sign_id")[["Sign-Arabic", "Sign-English"]].first().reset_index()
cm = orig.rename(columns={"SignID": "sign_id", "Sign-Arabic": "arabic_original", "Sign-English": "english_original"})
cm = cm.merge(
    rev.rename(columns={"Sign-Arabic": "arabic_reviewed", "Sign-English": "english_reviewed"}),
    on="sign_id", how="left", validate="one_to_one")
cm["label_changed"] = (
    (cm.english_original.astype(str) != cm.english_reviewed.astype(str))
    | (cm.arabic_original.astype(str) != cm.arabic_reviewed.astype(str)))
cm["english_duplicate"] = cm.english_reviewed.astype(str).str.lower().duplicated(keep=False)
cm["language"] = "ArSL"
cm.to_csv(R / "class_mapping_v2.csv", index=False, encoding="utf-8-sig")
print("labels changed vs original:", int(cm.label_changed.sum()))
print(cm[cm.label_changed][["sign_id", "english_original", "english_reviewed"]].to_string(index=False))

cmap = cm.set_index("sign_id")
meta = pd.DataFrame({
    "sample_id": df.sample_id,
    "source_dataset": "KArSL-502",
    "source_sample_id": df.video_path_in_archive.str.split("/").str[-1],
    "sign_language": "ArSL",
    "original_label": df.sign_id_original,
    "class_label": df.sign_id_corrected,
    "english_label": df.sign_id_corrected.map(cmap.english_reviewed),
    "arabic_label": df.sign_id_corrected.map(cmap.arabic_reviewed),
    "signer_id": df.signer_id,
    "session_id": df.session_id,
    "archive_relpath": df.archive_path.str.replace("/content/drive/MyDrive/KArSL-502/", "", regex=False),
    "video_path_in_archive": df.video_path_in_archive,
    "frames": df.frames,
    "fps": df.fps,
    "duration_sec": df.duration_sec,
    "width": df.width,
    "height": df.height,
    "author_split": df.split,
    "license_id": "NOT_STATED",
    "consent_ref": "NOT_STATED",
    "qc_status": df.qc_status,
    "qc_flags": df.qc_flags,
    "qc_reason": df.qc_reason,
    "usable": df.usable,
    "checksum_sha256": df.sha256,
})
meta.to_csv(R / "metadata_arsl.csv", index=False, encoding="utf-8-sig")

sources = pd.DataFrame([{
    "dataset_name": "KArSL-502",
    "official_url": "https://hamzah-luqman.github.io/KArSL/",
    "version_release": "KArSL-502 (502 signs); KArSL-100 and KArSL-190 are subsets",
    "access_date": today,
    "license_and_terms_url": "none stated on the official page",
    "permitted_use": "academic, non-commercial graduation project (team statement); citation of the source paper requested by the source page",
    "restrictions": "license not stated; redistribution of videos and derived data not verified",
    "sign_language_variety": "Arabic Sign Language (ArSL), not Egyptian Sign Language",
    "modality": "RGB video from Kinect V2 (color stream); depth and skeleton not obtained",
    "reported_counts": "502 signs x 3 signers x 50 repetitions = 75300",
    "verified_counts": f"{len(df)} videos in 30 archives; {len(u)} usable after QC",
    "missing_files": "no file missing from archives; per-signer count differences listed in count_anomalies.csv",
    "signer_metadata_available": "signer id from file names only; no demographic metadata obtained",
    "provenance_notes": "downloaded from the official Google Drive folder KArSL-502",
    "owner": "Dataset team (KArSL)",
    "approval_status": "pending project lead review; license unresolved",
}])
sources.to_csv(R / "dataset_sources.csv", index=False, encoding="utf-8-sig")

status = df.qc_status.value_counts()
reasons = df.qc_reason.str.split(";").explode()
reasons = reasons[reasons != ""].value_counts()
reason_text = "\n".join(f"- {k}: {v}" for k, v in reasons.items())
anom = pd.read_csv(R / "count_anomalies.csv")
big = anom[anom["diff"].abs() >= 10].sort_values(["signer_id", "sign_id"])
big_text = "\n".join(
    f"- {r.signer_id} sign {int(r.sign_id)}: {int(r.n)} videos (expected 50, {int(r['diff']):+d})"
    for _, r in big.iterrows())
sess = u.groupby(["signer_id", "batch", "recording_date"]).split.nunique()
shared = int((sess == 2).sum())
cc = u.groupby("class_label" if "class_label" in u else "sign_id_corrected").size()
per_signer = u.groupby("signer_id").size()
short = df.qc_flags.str.contains("TOO_SHORT")
short_by_signer = df[short].groupby("signer_id").size()
dec_fail = int((~df.decode_ok.astype(bool)).sum())
last_fail = int((~df.last_frame_ok.astype(bool)).sum())
n_dup_hash = int(df.sha256.duplicated(keep=False).sum())

qc_report = f"""# KArSL-502 Quality Control Report

Generated: {today}. Source: KArSL-502 (Arabic Sign Language), 3 signers, 502 classes.

## Summary
- Videos in archives: {len(df)}
- Usable videos: {len(u)}
- QC status: PASS {int(status.get('PASS', 0))}, REVIEW {int(status.get('REVIEW', 0))}, EXCLUDE {int(status.get('EXCLUDE', 0))}
- Usable per signer: {', '.join(f'{k} {v}' for k, v in per_signer.items())}
- Usable videos per class: min {int(cc.min())}, median {int(cc.median())}, max {int(cc.max())}

## Methods
- Archive integrity: 7z test on all 30 archives (30 passed).
- Video checks on every video: opens, first and last frame decode, frame count, fps, resolution, duration, SHA-256.
- Exact duplicates: SHA-256 comparison.
- Label checks: filename sign ID against folder, video counts per signer and sign against the expected 50, recording-time clusters, and visual checks by a team member. No sign-language expert was available.

## Results
- Decode failures: {dec_fail}. Last-frame failures: {last_fail}.
- Resolution 1920x1080 and 30 fps for all videos.
- Frames per video: min {int(df.frames.min())}, median {int(df.frames.median())}, max {int(df.frames.max())}.
- Videos in exact-duplicate groups: {n_dup_hash}.

## QC reasons (counts)
{reason_text}

## Exclusions and corrections
- S01 signs 27 and 28 (150 videos) are excluded and quarantined. Folder 0028 holds 50 byte-identical copies of 50 videos in folder 0027, and the other 50 videos in folder 0027 have unique hashes. Unresolved.
- S01 sign 5 to sign 6: 33 videos relabeled in sign_id_corrected (original kept). Evidence: recording-time gap, count arithmetic 17 + 33 = 50, visual check by a team member. No expert review.
- S01 sign 7: 33 files named 0001 are kept under folder ID 7 (count 33 + 17 = 50).
- 2 videos with fewer than 10 frames are excluded from use pending manual check.
- Short videos (fewer than 15 frames): {int(short.sum())} kept as PASS. By signer: {', '.join(f'{k} {v}' for k, v in short_by_signer.items())}. Treated as signer style, mostly S02 on letter signs.

## Count differences (raw counts before correction, difference of 10 or more)
{big_text}
- All differences are in count_anomalies.csv. Missing recordings are treated as missing data, not removed.

## Splits
- author_split (from the source): same signers and sessions appear in train and test. {shared} of {len(sess)} signer-batch-date groups appear in both. Do not use it alone to claim generalization.
- split_manifest_v2.csv: leave-one-signer-out columns for S01, S02 and S03. With S01 as test, classes 27 and 28 have no test samples (500 classes).

## Open issues
- License and terms not stated by the source.
- Quarantine of S01 signs 27 and 28 unresolved.
- No sign-language expert review of any label decision.
- Only 3 signers; no demographic metadata.
- Duplicate English labels: signs 213 and 214, signs 104 and 355. Use the sign ID as the class key.
"""
(R / "qc_report.md").write_text(qc_report, encoding="utf-8")

dataset_card = f"""# Dataset Card: KArSL-502 (SIGNOVA preparation)

## Intended use
Isolated Arabic Sign Language sign classification research in a graduation project. Not for commercial use. Not validated for continuous translation or interpreter-level reliability.

## Source and language
- Source: KArSL database, official page https://hamzah-luqman.github.io/KArSL/ (accessed {today}).
- Language: Arabic Sign Language (ArSL). It is not labeled or claimed as Egyptian Sign Language.
- Modality used: RGB video (Kinect V2). Depth and skeleton were not obtained.
- License: not stated on the source page. Citation of the source paper is requested.

## Composition
- {len(df)} videos, 502 classes, 3 signers (S01, S02, S03), 30 archives.
- Expected 50 repetitions per signer per sign (75300 total). Actual count differs, see count_anomalies.csv.
- 1920x1080 at 30 fps. Median 38 frames per video.
- File name format: batch_signer_signID_(day_month_year_hour_minute_second)_c.mp4. The signer is the second field and matches the folder. The meaning of the first field (batch) is inferred, not documented.

## Processing
- Metadata built from archive listings and file names. Original sample IDs and labels preserved.
- Full video QC (decode, duration, fps, resolution, SHA-256).
- Labels: sign ID is the class key. Original and reviewed labels are in class_mapping_v2.csv.
- sign_id_original is never overwritten. sign_id_corrected holds corrections.

## Splits
- author_split: train and test from the source. Same signers and sessions on both sides.
- split_manifest_v2.csv: leave-one-signer-out for S01, S02 and S03.

## Known issues
- S01 signs 27 and 28 quarantined (150 videos).
- S01 sign 5 and 6: 33 videos relabeled to 6 (visual check, no expert).
- S01 sign 7: 33 files with wrong filename sign ID, folder ID used.
- Some signs have 25 instead of 50 videos for one signer.
- Short videos (under 15 frames) are concentrated in S02 letter signs.

## Limitations
- Three signers only: no claim about unseen-signer generalization beyond leave-one-signer-out results.
- Labels were not reviewed by a sign-language expert.
- Only isolated signs. No continuous signing, no out-of-vocabulary signs.
- Archive-level checksums were not computed. Video-level SHA-256 is in metadata_arsl.csv.
"""
(R / "dataset_card.md").write_text(dataset_card, encoding="utf-8")

handoff = f"""# Handoff: KArSL-502 (dataset team to ML team)

Dataset version: KArSL-502 v1, metadata_arsl.csv, split_manifest_v2.csv. Prepared {today}. Approval by the project lead: pending.

## Files
- metadata_arsl.csv: one row per video with labels, QC status and usable flag.
- split_manifest_v2.csv: author_split and leave-one-signer-out columns for usable samples.
- class_mapping_v2.csv, qc_flags.csv, qc_report.md, count_anomalies.csv, duplicate_report.csv, label_issues.md, dataset_card.md, dataset_sources.csv, download_manifest.csv.

## How to load
1. Read metadata_arsl.csv and keep rows with usable == True ({len(u)} rows).
2. Use class_label as the class key (502 classes). original_label keeps the source value.
3. Use the split_manifest_v2.csv columns. loso_test_S01, loso_test_S02 and loso_test_S03 mark the test signer.
4. Videos are inside the .7z archives: archive_relpath plus video_path_in_archive. Raw videos are not in Git.
5. REVIEW samples (66 usable) can be included or excluded with qc_status.

## Constraints
- Do not use author_split alone for evaluation claims.
- With S01 as test, classes 27 and 28 have no test samples.
- Label decisions were not reviewed by a sign-language expert.
- License is not stated. Derived data may not be redistributed until it is clarified.
- Representation extraction (for example landmarks) is decided and documented by the ML team. This team made no such choice.
"""
(R / "handoff.md").write_text(handoff, encoding="utf-8")

for n in ["class_mapping_v2.csv", "metadata_arsl.csv", "dataset_sources.csv", "qc_report.md", "dataset_card.md", "handoff.md"]:
    print(n, round((R / n).stat().st_size / 1024, 1), "KB")
print(qc_report[:1500])

In [ ]:
import pandas as pd
from pathlib import Path

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
REVIEWER = "radwa sayed"
cm = pd.read_csv(R / "class_mapping_v2.csv")

ch = cm[cm.label_changed]
print("changed:", len(ch))
print("arabic text changed:", int((ch.arabic_original.astype(str) != ch.arabic_reviewed.astype(str)).sum()))
print(ch[["sign_id", "arabic_reviewed", "english_original", "english_reviewed"]].to_string(index=False))

cm["label_review_status"] = "original"
cm.loc[cm.label_changed, "label_review_status"] = "english_gloss_corrected_no_expert_review"
cm["label_reviewer"] = ""
cm.loc[cm.label_changed, "label_reviewer"] = REVIEWER
cm.to_csv(R / "class_mapping_v2.csv", index=False, encoding="utf-8-sig")
print(cm.label_review_status.value_counts())

add = """
## 7. English label corrections (48 classes)
- The reviewed label file corrects the English gloss of 48 classes. See class_mapping_v2.csv (label_changed, label_review_status).
- The sign ID is the class key, so these edits do not change which videos belong to which class.
- The corrections were made by a team member without sign-language expert or linguist review.
"""
p = R / "label_issues.md"
p.write_text(p.read_text(encoding="utf-8") + add, encoding="utf-8")

card = R / "dataset_card.md"
t = card.read_text(encoding="utf-8")
t = t.replace("## Known issues\n", "## Known issues\n- English glosses of 48 classes were corrected in class_mapping_v2.csv without expert review. The sign ID stays the class key.\n")
card.write_text(t, encoding="utf-8")
print(card.read_text(encoding="utf-8")[-900:])

In [ ]:
import pandas as pd
from pathlib import Path

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
cm = pd.read_csv(R / "class_mapping_v2.csv")

notes = {
    140: "Arabic word can mean pollination, fertilization or vaccination; gloss uncertain",
    391: "Arabic word can mean fiance or preacher; neighbouring signs are religious terms, so preacher is plausible",
    456: "Arabic word can mean duties or homework; gloss uncertain",
    475: "Arabic label covers emir and president; English gloss covers one meaning",
    478: "Arabic label covers crown prince and undersecretary; English gloss covers one meaning",
    499: "Same English gloss as sign 498; use class_name to tell them apart",
}

cm["label_note"] = cm.sign_id.map(notes).fillna("")
cm.loc[cm.sign_id.isin(notes), "label_review_status"] = "gloss_uncertain_needs_expert_check"
cm["class_name"] = cm.sign_id.astype(str).str.zfill(3) + "_" + cm.english_reviewed.astype(str).str.replace(" ", "_")

print("class_name unique:", cm.class_name.is_unique)
print(cm.label_review_status.value_counts())
cm.to_csv(R / "class_mapping_v2.csv", index=False, encoding="utf-8-sig")

In [ ]:
import shutil
import pandas as pd
from pathlib import Path
from datetime import date

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
meta = pd.read_csv(R / "metadata_arsl.csv", low_memory=False)
sm = pd.read_csv(R / "split_manifest_v2.csv")
cm = pd.read_csv(R / "class_mapping_v2.csv")
meta["qc_flags"] = meta.qc_flags.fillna("")
u = meta[meta.usable]

cols = ["sample_id", "qc_status", "qc_flags", "qc_reason", "usable", "signer_id",
        "original_label", "class_label", "author_split", "video_path_in_archive"]
flagged = meta[(meta.qc_status != "PASS") | (meta.qc_flags != "")]
flagged[cols].to_csv(R / "qc_flags.csv", index=False, encoding="utf-8-sig")
print("qc_flags rows:", len(flagged))

folds = ["loso_test_S01", "loso_test_S02", "loso_test_S03"]
tests = []

def add(name, ok, detail=""):
    tests.append({"test": name, "passed": bool(ok), "detail": str(detail)})

add("sample_id unique", meta.sample_id.is_unique)
add("row count is 75514", len(meta) == 75514, len(meta))
add("split v2 ids equal usable ids", set(sm.sample_id) == set(u.sample_id), f"{len(sm)} vs {len(u)}")
add("each sample in exactly one LOSO test fold", ((sm[folds] == "test").sum(axis=1) == 1).all())
add("no duplicate hash among usable", not u.checksum_sha256.duplicated().any())
add("502 classes in class mapping", set(meta.class_label) == set(cm.sign_id) and len(cm) == 502)
add("class_name unique", cm.class_name.is_unique)
add("original label equals folder sign id", (meta.video_path_in_archive.str[:4].astype(int) == meta.original_label).all())
add("EXCLUDE implies not usable", meta[(meta.qc_status == "EXCLUDE") & meta.usable].empty)

rep = pd.DataFrame(tests)
rep["dataset_version"] = "KArSL-502 v1 / split v2"
rep["run_date"] = date.today().isoformat()
rep.to_csv(R / "dataset_tests_report.csv", index=False)
print(rep[["test", "passed", "detail"]].to_string(index=False))

H = R.parent / "handoff"
H.mkdir(exist_ok=True)
files = ["dataset_sources.csv", "download_manifest.csv", "metadata_arsl.csv", "class_mapping_v2.csv",
         "qc_report.md", "qc_flags.csv", "split_manifest_v2.csv", "dataset_card.md", "handoff.md",
         "label_issues.md", "count_anomalies.csv", "duplicate_report.csv", "dataset_tests_report.csv"]
for f in files:
    shutil.copy(R / f, H / f)
print(sorted(x.name for x in H.iterdir()))

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd

drive.mount("/content/drive")

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
H = R.parent / "handoff"

required = [
    "dataset_sources.csv",
    "download_manifest.csv",
    "metadata_arsl.csv",
    "class_mapping_v2.csv",
    "qc_report.md",
    "qc_flags.csv",
    "split_manifest_v2.csv",
    "dataset_card.md",
    "handoff.md",
    "dataset_tests_report.csv"
]

print("=== REQUIRED FILES ===")
for name in required:
    print(f"{name}: {'READY' if (R / name).exists() else 'MISSING'}")

print("\n=== HANDOFF FILES ===")
for name in required:
    print(f"{name}: {'READY' if (H / name).exists() else 'MISSING'}")

m = pd.read_csv(R / "metadata_arsl.csv", low_memory=False)
s = pd.read_csv(R / "split_manifest_v2.csv")
t = pd.read_csv(R / "dataset_tests_report.csv")

usable = m["usable"].astype(str).str.lower().eq("true")

print("\n=== DATASET SUMMARY ===")
print("Total samples:", len(m))
print("Usable samples:", usable.sum())
print("Classes:", m["class_label"].nunique())
print("Signers:", m["signer_id"].nunique())
print("QC status:\n", m["qc_status"].value_counts().to_string())

print("\n=== DATASET TESTS ===")
print(t[["test", "passed"]].to_string(index=False))

print("\n=== SPLIT VALIDATION ===")
print("Unique sample IDs:", m["sample_id"].is_unique)
print("Usable samples match split manifest:",
      set(m.loc[usable, "sample_id"]) == set(s["sample_id"]))
print("Duplicate split IDs:", s["sample_id"].duplicated().sum())

print("\n=== SOURCE REGISTER ===")
sources = pd.read_csv(R / "dataset_sources.csv")
print(sources.to_string(index=False, max_colwidth=55))

print("\n=== REMAINING REVIEW CASES ===")
print(m.loc[m["qc_status"] == "REVIEW", "qc_reason"]
      .fillna("UNSPECIFIED").value_counts().to_string())

print("\n=== FINAL STATUS ===")
print("Missing required files:", sum(not (R / x).exists() for x in required))
print("Failed recorded tests:", (~t["passed"].astype(str).str.lower().eq("true")).sum())

In [ ]:
import pandas as pd
import shutil
from pathlib import Path

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
H = R.parent / "handoff"

m = pd.read_csv(R / "metadata_arsl.csv", low_memory=False)
s = pd.read_csv(R / "split_manifest_v2.csv")

m["usable"] = m["usable"].astype(str).str.lower().eq("true")

clean = m[(m.qc_status == "PASS") & m.usable].copy()
review = m[m.qc_status == "REVIEW"].copy()

clean_split = s[s.sample_id.isin(clean.sample_id)].copy()

checks = {
    "Unique sample IDs": clean.sample_id.is_unique,
    "No duplicate hashes": clean.checksum_sha256.is_unique,
    "Clean manifest matches PASS samples": set(clean_split.sample_id) == set(clean.sample_id),
    "All clean samples are PASS": clean.qc_status.eq("PASS").all(),
    "All clean samples are usable": clean.usable.all(),
    "No signer leakage in LOSO": all(
        set(clean_split.loc[clean_split[f"loso_test_{x}"] == "test", "signer_id"]) == {x}
        and not clean_split.loc[
            clean_split[f"loso_test_{x}"] == "train", "signer_id"
        ].eq(x).any()
        for x in ["S01", "S02", "S03"]
    )
}

pd.DataFrame([
    {"check": k, "passed": v}
    for k, v in checks.items()
]).to_csv(R / "final_readiness_checks.csv", index=False)

if not all(checks.values()):
    raise ValueError("Final validation failed. Existing files were not changed.")

clean_split.to_csv(R / "split_manifest_pass_only.csv", index=False)

review[[
    "sample_id", "signer_id", "original_label",
    "class_label", "qc_status", "qc_reason", "usable"
]].assign(
    review_status="PENDING",
    reviewer_decision=""
).to_csv(R / "open_review_cases.csv", index=False)

for name in [
    "split_manifest_pass_only.csv",
    "final_readiness_checks.csv",
    "open_review_cases.csv"
]:
    shutil.copy2(R / name, H / name)

print("=== FINAL QC SUMMARY ===")
print("Total samples:", len(m))
print("PASS:", len(clean))
print("REVIEW:", len(review))
print("EXCLUDE:", (m.qc_status == "EXCLUDE").sum())
print("Clean classes:", clean.class_label.nunique())
print("Clean signers:", clean.signer_id.nunique())

print("\n=== OPEN REVIEW CASES ===")
print(review.qc_reason.value_counts().to_string())

print("\n=== FINAL CHECKS ===")
for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

print("\n=== FILES CREATED ===")
for name in [
    "split_manifest_pass_only.csv",
    "final_readiness_checks.csv",
    "open_review_cases.csv"
]:
    print(name, "READY" if (H / name).exists() else "MISSING")

In [ ]:
import pandas as pd
import shutil
from pathlib import Path

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
H = R.parent / "handoff"
H.mkdir(exist_ok=True)

s = pd.read_csv(R / "split_manifest_pass_only.csv")

folds = [
    ("fold_1", "S03", "S02", "S01"),
    ("fold_2", "S01", "S03", "S02"),
    ("fold_3", "S02", "S01", "S03")
]

parts = []
results = []

for name, train, val, test in folds:
    d = s[["sample_id", "signer_id", "sign_id"]].copy()
    d["fold"] = name
    d["split"] = d.signer_id.map({
        train: "train",
        val: "validation",
        test: "test"
    })
    d["split_version"] = "v3_pass_only"
    parts.append(d)

    for split in ["train", "validation", "test"]:
        x = d[d.split == split]
        results.append({
            "fold": name,
            "split": split,
            "signer": x.signer_id.iloc[0],
            "samples": len(x),
            "classes": x.sign_id.nunique()
        })

out = pd.concat(parts, ignore_index=True)
report = pd.DataFrame(results)

checks = {
    "All samples assigned": all(len(d) == len(s) for d in parts),
    "No duplicate IDs": not out.duplicated(["fold", "sample_id"]).any(),
    "No missing splits": out.split.isna().sum() == 0,
    "No signer leakage": all(
        d.groupby("signer_id").split.nunique().max() == 1
        for d in parts
    ),
    "All classes represented": (report.classes == 502).all()
}

print("=== SPLIT DISTRIBUTION ===")
print(report.to_string(index=False))

print("\n=== VALIDATION RESULTS ===")
for name, result in checks.items():
    print(f"{name}: {'PASS' if result else 'FAIL'}")

if all(checks.values()):
    out.to_csv(R / "split_manifest_3way.csv", index=False)
    report.to_csv(R / "split_distribution.csv", index=False)

    for name in ["split_manifest_3way.csv", "split_distribution.csv"]:
        shutil.copy2(R / name, H / name)

    print("\n=== FINAL STATUS ===")
    print("Total clean samples:", len(s))
    print("Number of folds:", len(folds))
    print("Files saved successfully: YES")
else:
    print("\nValidation failed. No files saved.")

In [ ]:
import pandas as pd
import shutil
from pathlib import Path

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
H = R.parent / "handoff"
H.mkdir(parents=True, exist_ok=True)

s = pd.read_csv(
    R / "split_manifest_pass_only.csv",
    dtype={"sample_id": str, "signer_id": str, "sign_id": str}
)

signers = ["S01", "S02", "S03"]
classes = {
    x: set(s.loc[s.signer_id == x, "sign_id"])
    for x in signers
}

common = set.intersection(*classes.values())
all_classes = set.union(*classes.values())

gaps = pd.DataFrame([
    {"signer_id": signer, "missing_sign_id": sign}
    for signer in signers
    for sign in sorted(all_classes - classes[signer])
])

clean = s[s.sign_id.isin(common)].copy()
excluded = s[~s.sign_id.isin(common)].copy()

folds = [
    ("fold_1", "S03", "S02", "S01"),
    ("fold_2", "S01", "S03", "S02"),
    ("fold_3", "S02", "S01", "S03")
]

parts = []
results = []

for name, train, val, test in folds:
    d = clean.copy()
    d["fold"] = name
    d["split"] = d.signer_id.map({
        train: "train",
        val: "validation",
        test: "test"
    })
    d["split_version"] = "v3_common_classes"
    parts.append(d)

    for split in ["train", "validation", "test"]:
        x = d[d.split == split]
        results.append({
            "fold": name,
            "split": split,
            "signer": x.signer_id.iloc[0],
            "samples": len(x),
            "classes": x.sign_id.nunique()
        })

out = pd.concat(parts, ignore_index=True)
report = pd.DataFrame(results)

checks = {
    "All clean samples assigned per fold": all(
        len(d) == len(clean) for d in parts
    ),
    "No duplicate sample IDs per fold": not out.duplicated(
        ["fold", "sample_id"]
    ).any(),
    "No missing splits": out.split.isna().sum() == 0,
    "No signer leakage": all(
        d.groupby("signer_id").split.nunique().max() == 1
        for d in parts
    ),
    "Same classes in every split": all(
        set(d.loc[d.split == part, "sign_id"]) == common
        for d in parts
        for part in ["train", "validation", "test"]
    ),
    "Original samples preserved": len(clean) + len(excluded) == len(s)
}

print("=== CLASS COVERAGE ===")
print("Original classes:", len(all_classes))
print("Common classes:", len(common))
print("Missing class records:")
print(gaps.to_string(index=False))
print("Samples outside common classes:", len(excluded))
print("Final samples per fold:", len(clean))

print("\n=== SPLIT DISTRIBUTION ===")
print(report.to_string(index=False))

print("\n=== VALIDATION RESULTS ===")
for name, result in checks.items():
    print(f"{name}: {'PASS' if result else 'FAIL'}")

if all(checks.values()):
    files = {
        "split_manifest_3way.csv": out,
        "split_distribution.csv": report,
        "split_class_gaps.csv": gaps,
        "split_excluded_classes.csv": excluded
    }

    for name, df in files.items():
        df.to_csv(R / name, index=False)
        shutil.copy2(R / name, H / name)

    print("\n=== FINAL STATUS ===")
    print("All checks passed: YES")
    print("Files saved:", len(files))
    print("Dataset ready for baseline experiments: YES")
    print("License approval: PENDING")
else:
    print("\nValidation failed. No files saved.")

In [ ]:
import pandas as pd
import shutil
from pathlib import Path
from datetime import date

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
H = R.parent / "handoff"
H.mkdir(exist_ok=True)

m = pd.read_csv(R / "metadata_arsl.csv", low_memory=False)
s = pd.read_csv(R / "split_manifest_3way.csv")
cm = pd.read_csv(R / "class_mapping_v2.csv")

clean = m[m.qc_status.eq("PASS") & m.usable.eq(True)].copy()
active_ids = sorted(set(s.sign_id))
active = clean[clean.class_label.isin(active_ids)].copy()

mapping = cm[cm.sign_id.isin(active_ids)].copy()
mapping = mapping.sort_values("sign_id")
mapping.insert(0, "model_index", range(len(mapping)))

checks = {
    "500 active classes": len(active_ids) == 500,
    "75095 active samples": len(active) == 75095,
    "All samples PASS": active.qc_status.eq("PASS").all(),
    "No duplicate hashes": active.checksum_sha256.is_unique,
    "Unique class mapping": mapping.sign_id.is_unique,
    "All labels mapped": set(active.class_label) == set(mapping.sign_id),
    "Three evaluation folds": s.fold.nunique() == 3,
    "No duplicate IDs within folds": not s.duplicated(["fold", "sample_id"]).any(),
    "Exact sample coverage": all(
        set(g.sample_id) == set(active.sample_id)
        for _, g in s.groupby("fold")
    ),
    "Signer independent splits": all(
        g.groupby("signer_id").split.nunique().max() == 1
        for _, g in s.groupby("fold")
    ),
    "500 classes in each split": all(
        g.sign_id.nunique() == 500
        for _, g in s.groupby(["fold", "split"])
    ),
    "No missing labels": not s.sign_id.isna().any()
}

tests = pd.DataFrame([
    {"test": k, "passed": bool(v)}
    for k, v in checks.items()
])

print("=== FINAL VERSION 3 TESTS ===")
print(tests.to_string(index=False))

if not all(checks.values()):
    raise ValueError("Final validation failed. No files changed.")

mapping.to_csv(R / "class_mapping_v3.csv", index=False, encoding="utf-8-sig")
tests.to_csv(R / "dataset_tests_v3.csv", index=False)

summary = f"""# KArSL-502 Benchmark Version 3

Date: {date.today().isoformat()}
Dataset: KArSL-502
Language: Arabic Sign Language (ArSL), not Egyptian Sign Language
Original samples: {len(m)}
Original classes: {m.class_label.nunique()}
PASS samples before class filtering: {len(clean)}
Benchmark samples: {len(active)}
Benchmark classes: {len(active_ids)}
Signers: {active.signer_id.nunique()}
QC REVIEW samples: {(m.qc_status == "REVIEW").sum()}
QC EXCLUDE samples: {(m.qc_status == "EXCLUDE").sum()}
PASS samples outside benchmark: {len(clean) - len(active)}

## Split Protocol
Three signer-independent train/validation/test folds.
One signer is assigned to each partition per fold.
All 500 benchmark classes occur in each partition.
There is no sample or signer overlap within a fold.
The same samples occur in different roles across folds.
Model selection must use validation data only.
Final test results must not be used for tuning.

## Limitations
Only three signers are available.
Each fold trains on one signer.
Classes 27 and 28 are excluded from this benchmark.
Original metadata and the 502-class mapping remain preserved.
68 QC REVIEW cases remain unresolved.
Some label corrections lack sign-language expert validation.
The source license is unresolved.
Project-lead approval and usage-rights verification are pending.

## Benchmark Files
split_manifest_3way.csv
split_distribution.csv
split_class_gaps.csv
split_excluded_classes.csv
split_manifest_pass_only.csv
class_mapping_v3.csv
dataset_tests_v3.csv

## Status
Technical data and split checks: PASS
License approval: PENDING
Expert label review: PENDING
Final project-lead acceptance: PENDING
"""

(R / "benchmark_v3_summary.md").write_text(summary, encoding="utf-8")

section = """## Version 3 Benchmark Update

See benchmark_v3_summary.md for the final PASS-only benchmark.
The approved-by-tests subset contains 75,095 samples and 500 classes.
Three signer-independent folds contain train, validation and test partitions.
Classes 27 and 28 are outside this benchmark.
The 502-class original dataset remains preserved.
The benchmark is technically validated, not legally approved for use.
License, expert review and project-lead approval remain pending.
"""

for name in ["qc_report.md", "dataset_card.md", "handoff.md"]:
    p = R / name
    text = p.read_text(encoding="utf-8")
    text = text.split("## Version 3 Benchmark Update")[0].rstrip()
    p.write_text(text + "\n\n" + section, encoding="utf-8")

files = [
    "class_mapping_v3.csv",
    "dataset_tests_v3.csv",
    "benchmark_v3_summary.md",
    "qc_report.md",
    "dataset_card.md",
    "handoff.md"
]

for name in files:
    shutil.copy2(R / name, H / name)

print("\n=== DATASET SUMMARY ===")
print("Original samples:", len(m))
print("Benchmark samples:", len(active))
print("Benchmark classes:", len(mapping))
print("Evaluation folds:", s.fold.nunique())

print("\n=== HANDOFF STATUS ===")
for name in files:
    print(name, "READY" if (H / name).exists() else "MISSING")

print("\n=== FINAL RESULT ===")
print("All technical checks passed:", all(checks.values()))
print("License status: PENDING")
print("Expert review: PENDING")
print("Project lead approval: PENDING")